#RNN


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
sentences = [
    'I love my dog',
    'I love my cat',
    'You love my dog!',
    'Do you think my dog is amazing?' ,
    'I do not like my cat',
    'I do not like my dog',
    'I love dogs',
    'I love to play cricket with my friends',
'Artificial Intelligence is changing the world',
'Deep learning models require large datasets' ,
'Natural language processing helps computers understand text',
'Recurrent neural networks process sequential data',
'Machine learning is a fascinating field',
'Data science combines statistics and programming',
'The sun rises in the east every morning',
'She likes to drink coffee in the morning',
'We are learning neural networks in college',
]
labels = [1]*15 + [0]*15
labels = np.array(labels)
print(labels)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [14]:
vocab_size = 2000
tokinizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokinizer.fit_on_texts(sentences)
word_index = tokinizer.word_index
sequences = tokinizer.texts_to_sequences(sentences)
# The model expects input shape (None, 8) because `maxlen` was 8 initially.
# However, X was padded to `maxlen=10`.
# To resolve the mismatch, we ensure the `maxlen` variable used for the Input layer
# is consistent with the padding length of X.
maxlen = 10 # Explicitly set maxlen to 10 to match the padding length of X
print(maxlen)
X = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
y = labels

10


In [4]:
X[0]

array([3, 4, 2, 5, 0, 0, 0, 0, 0, 0], dtype=int32)

In [5]:
X

array([[ 3,  4,  2,  5,  0,  0,  0,  0,  0,  0],
       [ 3,  4,  2, 11,  0,  0,  0,  0,  0,  0],
       [12,  4,  2,  5,  0,  0,  0,  0,  0,  0],
       [ 7, 12, 20,  2,  5,  8, 21,  0,  0,  0],
       [ 3,  7, 13, 14,  2, 11,  0,  0,  0,  0],
       [ 3,  7, 13, 14,  2,  5,  0,  0,  0,  0],
       [ 3,  4, 22,  0,  0,  0,  0,  0,  0,  0],
       [ 3,  4, 15, 23, 24, 25,  2, 26,  0,  0],
       [27, 28,  8, 29,  6, 30,  0,  0,  0,  0],
       [31,  9, 32, 33, 34, 35,  0,  0,  0,  0],
       [36, 37, 38, 39, 40, 41, 42,  0,  0,  0],
       [43, 16, 17, 44, 45, 18,  0,  0,  0,  0],
       [46,  9,  8, 47, 48, 49,  0,  0,  0,  0],
       [18, 50, 51, 52, 53, 54,  0,  0,  0,  0],
       [ 6, 55, 56, 10,  6, 57, 58, 19,  0,  0],
       [59, 60, 15, 61, 62, 10,  6, 19,  0,  0],
       [63, 64,  9, 16, 17, 10, 65,  0,  0,  0]], dtype=int32)

In [6]:
embeb_dim = 16
rnn_units = 8

In [17]:
from tensorflow.keras.layers import Input, SimpleRNN, Embedding, Dense
from tensorflow.keras.models import Model

inp = Input(shape= (maxlen,),dtype="int32",name = 'input')
x = Embedding(input_dim=vocab_size,output_dim=embeb_dim,mask_zero=True,name = 'embed')(inp)
rnn = SimpleRNN(units = rnn_units, return_sequences = False,return_state= False,name = 'simple_rnn')
x_last = rnn(x)
out = Dense(1,activation='sigmoid',name = 'out')(x_last)
model = Model(inputs =inp,outputs = out)
model.compile(optimizer='adam',loss = 'binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 10, 16)    │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 10)        │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(X,y[:len(X)],epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2941 - loss: 0.7164
Epoch 2/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7059 - loss: 0.6747
Epoch 3/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8824 - loss: 0.6434
Epoch 4/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.6126
Epoch 5/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.5832
Epoch 6/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.5547
Epoch 7/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.5272
Epoch 8/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.5007
Epoch 9/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.4746
Epoch 10/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.4494
Epoch 11/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.4258
Epoch 12/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.4020
E

In [19]:

from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))


Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Sentence: I love my dog
Token ids: [[3 4 2 5 0 0 0 0 0 0]]
Hidden states per timestep shape: (1, 10, 8)
Hidden states (timesteps x units):
[[ 0.232  0.217 -0.021  0.117  0.029  0.075  0.16  -0.091]
 [-0.053 -0.078  0.082  0.381  0.07   0.116  0.317  0.151]
 [-0.339  0.204 -0.067  0.064  0.508 -0.118  0.26   0.236]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]
 [ 0.581  0.066 -0.167  0.213  0.356 -0.459  0.578 -0.055]]
